# 1 - Preprocessing the cAPTure dataset for IDS training/testing and scenario reconstruction

## 1.1 - Creating machine learning datasets

### 1.1.1 - Validate files and anchor file paths

In [1]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
print(f"working directory:\n\n{os.getcwd()}")
print("\ndirectory structure of data/cAPTure:\n")
for root, dirs, files in os.walk(PROJECT_ROOT / "data" / "cAPTure"):
        print(root+":")
        if dirs:
            print('\n'.join(dirs))
        if files:
            print('\n'.join(files))

working directory:

/home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/jupyter notebook

directory structure of data/cAPTure:

/home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/data/cAPTure:
attack
normal
merged
/home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/data/cAPTure/attack:
dollar_char_test
dollar_char_train
empty_conn_test
empty_conn_train
pub_exf_test
pub_exf_train
qos_mid_test
qos_mid_train
slash_char_test
slash_char_train
sub_exf_test
sub_exf_train
user_prop_test
user_prop_train
.ipynb_checkpoints
/home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/data/cAPTure/attack/dollar_char_test:
.ipynb_checkpoints
dollar_char_test.npz
dollar_char_test.pkl
dollar_char_test.pcap
dollar_char_test.csv
dollar_char_test_labeled.csv
dollar_char_test_flows.csv
/home/users/davide.zhang/u

In [2]:
DATA_ROOT = PROJECT_ROOT / "data" / "cAPTure"
MERGED_DATA_ROOT = DATA_ROOT / "merged"
ML_DATA_ROOT = DATA_ROOT / "ml"
PLOTS_DIR = ML_DATA_ROOT / "plots"
FINAL_DATA_ROOT = DATA_ROOT / "final_datasets"

for path in (ML_DATA_ROOT, PLOTS_DIR, FINAL_DATA_ROOT):
    path.mkdir(parents=True, exist_ok=True)

campaign_names = ["dollar_char"]  # keep identical order to attack_normal_merging
campaign_ids = {name: idx for idx, name in enumerate(campaign_names)}

### 1.1.2 - Dataset loading

In [3]:
import pandas as pd
import numpy as np
df_train=pd.read_csv(MERGED_DATA_ROOT / "train.csv", low_memory=False)
df_test=pd.read_csv(MERGED_DATA_ROOT / "test.csv", low_memory=False)

/tmp/ipykernel_2624591/3224523833.py:3: DtypeWarning: Columns (0: client_fingerprint, 1: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train=pd.read_csv(MERGED_DATA_ROOT / "train.csv")
/tmp/ipykernel_2624591/3224523833.py:4: DtypeWarning: Columns (0: client_fingerprint, 1: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test=pd.read_csv(MERGED_DATA_ROOT / "test.csv")


In [4]:
df_test["index"] = np.arange(len(df_test))

### 1.1.3 - Dataset cleaning

In [ ]:
df_train.columns = df_train.columns.str.strip()
df_test.columns = df_test.columns.str.strip()
initial_columns = df_train.columns
for column in initial_columns:
    if any(keyword in column for keyword in ["id", "ip", "mac", "oui", "duration", "seen", "protocol", "Unnamed", "fingerprint", "timestamp", "phase", "step"]):
        df_train = df_train.drop(columns=column)
        df_test = df_test.drop(columns=column, errors="ignore")

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
import numpy as np
df_train = df_train.dropna(thresh=2, axis=1).dropna(thresh=2, axis=0)
# Do not derive a different feature set from evaluation data.
df_test = df_test.dropna(thresh=2, axis=0)

In [ ]:
num_cols = df_train.select_dtypes(include="number").columns
df_train[num_cols] = df_train[num_cols].replace([np.inf, -np.inf, np.nan], -1)

str_cols = df_train.select_dtypes(exclude="number").columns
df_train[str_cols] = df_train[str_cols].fillna(" ")

In [ ]:
num_cols = df_test.select_dtypes(include="number").columns
df_test[num_cols] = df_test[num_cols].replace([np.inf, -np.inf, np.nan], -1)

str_cols = df_test.select_dtypes(exclude="number").columns
df_test[str_cols] = df_test[str_cols].fillna(" ")

In [ ]:
print(df_train.isna().sum().sum())

In [ ]:
print(df_test.isna().sum().sum())

In [ ]:
THRESHOLD = 0.99

In [ ]:
low_var_cols = [col for col in df_train.columns
                if df_train[col].value_counts(normalize=True, dropna=False).iloc[0] >= THRESHOLD]
df_train = df_train.drop(columns=low_var_cols)
df_test = df_test.drop(columns=low_var_cols, errors="ignore")

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
from preprocessing.training import align_test_schema
df_test = align_test_schema(df_train, df_test)

In [ ]:
df_train["label"].value_counts()

In [ ]:
df_test["label"].value_counts()

In [ ]:
from preprocessing.training import filter_known_test_classes
df_test = filter_known_test_classes(df_train, df_test)

In [ ]:
df_train["label"].value_counts()

In [ ]:
df_test["label"].value_counts()

### 1.1.7 - Saving train-set and test-set

In [ ]:
ML_DATA_ROOT.mkdir(parents=True, exist_ok=True)
df_train.to_csv(ML_DATA_ROOT / "ML_train.csv", index=False)
df_test.to_csv(ML_DATA_ROOT / "ML_test.csv", index=False)

## 1.2 - Computing anomaly score

### 1.2.1 - Loading the datasets

In [ ]:
import numpy as np
import pandas as pd

df_train = pd.read_csv(ML_DATA_ROOT / "ML_train.csv", low_memory=False)
df_test = pd.read_csv(ML_DATA_ROOT / "ML_test.csv", low_memory=False)
ind_test = df_test["index"].to_numpy()

### 1.2.2 - Categorical encoding for anomaly detection

In [ ]:
attack_labels = sorted(label for label in df_train["label"].unique() if label != "normal")
print("attack_labels:", attack_labels)

In [ ]:
from preprocessing.training import encode_binary_labels
label_encode = encode_binary_labels

In [ ]:
from preprocessing.training import apply_label_encoding
label_encode_categories = apply_label_encoding

In [ ]:
print(df_train.isna().sum().sum())

In [ ]:
print(df_test.isna().sum().sum())

In [ ]:
from preprocessing.training import split_training_validation

# Reproducible calibration holdout from training data; final test data is untouched.
df_fit, df_validation = split_training_validation(df_train, attack_labels)
df_train_encoded, categories = label_encode(df_fit, attack_labels)
df_validation_encoded = label_encode_categories(df_validation, attack_labels, categories)
df_test_encoded = label_encode_categories(df_test, attack_labels, categories)

In [ ]:
df_train_encoded.info()

In [ ]:
df_test_encoded.info()

In [ ]:
print(df_train_encoded.isna().sum().sum())

In [ ]:
print(df_test_encoded.isna().sum().sum())

### 1.2.3 - Feature label separation

In [ ]:
y_train, X_train = df_train_encoded["label"], df_train_encoded.drop("label", axis=1)
y_validation, X_validation = df_validation_encoded["label"], df_validation_encoded.drop("label", axis=1)
y_test, X_test = df_test_encoded["label"], df_test_encoded.drop(["label", "index"], axis=1)
feature_names = list(X_train.columns)
X_train.info()
X_validation.info()
X_test.info()

### 1.2.4 - Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_validation = scaler.transform(X_validation)
X_test = scaler.transform(X_test)

### 1.2.5 - Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor
regressor=RandomForestRegressor(random_state=42, n_jobs=-1)
regressor.fit(X_train, y_train)

In [ ]:
for col, importance in zip(feature_names, regressor.feature_importances_):
    print(f"{col}: {np.round(importance, 3)}")

### 1.2.7 - Computing contamination

In [ ]:
raw_contamination = float((y_train == 1).mean())
contamination = min(raw_contamination, 0.5)  # PyOD requires contamination in (0, 0.5]
print(f"raw contamination: {raw_contamination:.6f}")
print(f"clamped contamination: {contamination:.6f}")

### 1.2.8 - Train anomaly detectors

In [ ]:
from pyod.models.iforest import IForest
iforest = IForest(contamination=contamination, n_estimators=500, n_jobs=10, random_state=42)
iforest.fit(X_train)

In [ ]:
from pyod.models.hbos import HBOS
hbos = HBOS(n_bins=500, contamination=contamination)
hbos.fit(X_train)

In [ ]:
from pyod.models.sampling import Sampling
sampling = Sampling(contamination=contamination, subset_size=200, random_state=42)
sampling.fit(X_train)

In [ ]:
from sklearn.naive_bayes import GaussianNB
naive_bayes=GaussianNB()
naive_bayes.fit(X_train, y_train)

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
qda = QuadraticDiscriminantAnalysis(reg_param=1.0)
qda.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic_regression=LogisticRegression(max_iter=500, random_state=42)
logistic_regression.fit(X_train, y_train)

In [ ]:
from xgboost import XGBClassifier
xgb=XGBClassifier(n_estimators=50, random_state=42)
xgb.fit(X_train,y_train)

### 1.2.9 - Test anomaly detection function definition

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
def evaluate_metrics(predictor, name):
    y_pred=predictor.predict(X_test)
    acc=accuracy_score(y_test, y_pred)
    print(f"{name} test accuracy: {acc:.3f}")
    rec=recall_score(y_test, y_pred)
    print(f"{name} test recall: {rec:.3f}")
    prec=precision_score(y_test, y_pred)
    print(f"{name} test precision: {prec:.3f}")

### 1.2.10 - ROC and Precision-Recall Functions definition

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

def show_roc(predictor, name):
    y_prob = predictor.predict_proba(X_test)[:, 1]
    if len(np.unique(y_test)) < 2:
        print(f"Skipping ROC for {name}: fewer than 2 unique labels in y_test")
        return y_prob

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))

    RocCurveDisplay(
        fpr=fpr,
        tpr=tpr,
        roc_auc=roc_auc,
        estimator_name=name
    ).plot(ax=ax)

    ax.set_title(name)
    ax.grid(True)

    fig.suptitle(f"ROC Curve ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOTS_DIR / f"roc_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    return y_prob

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    PrecisionRecallDisplay,
)
import matplotlib.pyplot as plt

def show_prc(predictor, name):
    y_prob = predictor.predict_proba(X_test)[:, 1]
    if len(np.unique(y_test)) < 2:
        print(f"Skipping PRC for {name}: fewer than 2 unique labels in y_test")
        return y_prob

    pre, rec, _ = precision_recall_curve(y_test, y_prob)
    avg_pre = average_precision_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))

    PrecisionRecallDisplay(
        precision=pre,
        recall=rec,
        average_precision=avg_pre,
        estimator_name=name
    ).plot(ax=ax)

    ax.set_title(name)
    ax.grid(True)

    fig.suptitle(f"Precision-Recall Curve ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOTS_DIR / f"prc_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    return y_prob

In [ ]:
from preprocessing.training import calibrate_threshold

def compute_thresh(predictor, recall, name=None):
    threshold = calibrate_threshold(predictor, X_validation, y_validation, recall)
    prefix = f"{name} " if name else ""
    print(f"{prefix}computed threshold (recall={recall}): {threshold:.6f}")
    return threshold

### 1.2.11 - Test anomaly detectors

In [ ]:
evaluate_metrics(iforest, "IForest")

In [ ]:
show_roc(iforest, "IForest")

In [ ]:
show_prc(iforest, "IForest")

In [ ]:
evaluate_metrics(hbos, "HBOS")

In [ ]:
show_roc(hbos, "HBOS")

In [ ]:
show_prc(hbos, "HBOS")

In [ ]:
evaluate_metrics(sampling, "Sampling")

In [ ]:
show_roc(sampling, "Sampling")

In [ ]:
show_prc(sampling, "Sampling")

In [ ]:
evaluate_metrics(naive_bayes, "GaussianNB")

In [ ]:
show_roc(naive_bayes, "GaussianNB")

In [ ]:
show_prc(naive_bayes, "GaussianNB")

In [ ]:
evaluate_metrics(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
show_roc(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
show_prc(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
evaluate_metrics(logistic_regression, "LogisticRegression")

In [ ]:
show_roc(logistic_regression, "LogisticRegression")

In [ ]:
show_prc(logistic_regression, "LogisticRegression")

In [ ]:
evaluate_metrics(xgb, "XGBClassifier")

In [ ]:
show_roc(xgb, "XGBClassifier")

In [ ]:
show_prc(xgb, "XGBClassifier")

### 1.2.12 - Computing target threshold

In [ ]:
import json

target_recall = 0.8
anomaly_detectors = {
    "logistic_regression": logistic_regression,
    "xgb": xgb,
    "qda": qda,
    "naive_bayes": naive_bayes,
    "sampling": sampling,
    "hbos": hbos,
    "iforest": iforest,
}

calibrated_thresholds = {
    name: compute_thresh(model, target_recall, name=name)
    for name, model in anomaly_detectors.items()
}

ML_DATA_ROOT.mkdir(parents=True, exist_ok=True)
with open(ML_DATA_ROOT / "calibrated_thresholds.json", "w", encoding="utf-8") as f:
    json.dump({"target_recall": target_recall, "thresholds": calibrated_thresholds}, f, indent=2)

calibrated_thresholds

### 1.2.13 - Computing anomaly score 

In [ ]:
anomaly_score_dict = {
    name: model.predict_proba(X_test)[:, 1]
    for name, model in anomaly_detectors.items()
}
anomaly_score_dict["perfect"] = y_test.to_numpy()

In [ ]:
for key, df in anomaly_score_dict.items():
    print(key+ f": {df.size}")

## 1.3 - Computing attack type score

### 1.3.1 - Loading the datasets

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.training import (
    encode_binary_labels as label_encode,
    apply_label_encoding as label_encode_categories,
)

df_train = pd.read_csv(ML_DATA_ROOT / "ML_train.csv", low_memory=False)
df_test = pd.read_csv(ML_DATA_ROOT / "ML_test.csv", low_memory=False)
ind_test = df_test["index"].to_numpy()

### 1.3.2 - Categorical encoding for type mapping

In [ ]:
df_dict_train = {}
df_dict_test = {}
attack_types = sorted(label for label in df_train["label"].unique() if label != "normal")
print("attack_types:", attack_types)
for label in attack_types:
    df_dict_train[label], categories = label_encode(df_train, [label])
    df_dict_test[label] = label_encode_categories(df_test, [label], categories)

In [ ]:
for label in attack_types:
    print(label)
    print("train set")
    print(df_dict_train[label]["label"].value_counts())
    print("test set")
    print(df_dict_test[label]["label"].value_counts())

### 1.3.4 - Feature label separation

In [ ]:
y_train_dict={}
X_train_dict={}
y_test_dict={}
X_test_dict={}

for label in attack_types:
    y_train_dict[label], X_train_dict[label]= df_dict_train[label]["label"], df_dict_train[label].drop("label", axis=1)
    y_test_dict[label], X_test_dict[label]= df_dict_test[label]["label"], df_dict_test[label].drop(["label", "index"], axis=1)
    X_train_dict[label].info()
    X_test_dict[label].info()

### 1.3.5 - Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
for label in attack_types:
    scaler = StandardScaler()
    scaler.fit(X_train_dict[label])
    X_train_dict[label] = scaler.transform(X_train_dict[label])
    X_test_dict[label] = scaler.transform(X_test_dict[label])

### 1.3.5 - Train type mappers

In [ ]:
from xgboost import XGBClassifier
xgb_dict={}
for label in attack_types:
    xgboost=XGBClassifier(n_estimators=50, random_state=42)
    xgboost.fit(X_train_dict[label], y_train_dict[label])
    xgb_dict[label]=xgboost

In [ ]:
from sklearn.naive_bayes import GaussianNB
naive_bayes_dict={}
for label in attack_types:
    nb=GaussianNB()
    nb.fit(X_train_dict[label], y_train_dict[label])
    naive_bayes_dict[label]=nb

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# scikit-learn 1.8.0 requires solver='eigen' with shrinkage when a rare attack
# class (e.g., scp_inst) has fewer training samples than features; solver='svd'
# with reg_param=1.0 raises LinAlgError in that regime.
qda_dict = {}
for label in attack_types:
    qda = QuadraticDiscriminantAnalysis(solver="eigen", shrinkage=1.0)
    qda.fit(X_train_dict[label], y_train_dict[label])
    qda_dict[label] = qda

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic_regression_dict={}
for label in attack_types:
    logistic_regression=LogisticRegression(max_iter=500, random_state=42)
    logistic_regression.fit(X_train_dict[label], y_train_dict[label])
    logistic_regression_dict[label]=logistic_regression

### 1.3.6 - Test type mapper function definition

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

def evaluate_mapper(mapper_dict, name):
    for label, model in mapper_dict.items():
        y_true = y_test_dict[label]
        y_pred = model.predict(X_test_dict[label])
        acc = accuracy_score(y_true, y_pred)
        if len(np.unique(y_true)) < 2:
            print(f"{name} test accuracy for {label}: {acc:.3f} (fewer than 2 unique labels in test set; recall/precision skipped)")
            continue
        rec = recall_score(y_true, y_pred, zero_division=0)
        prec = precision_score(y_true, y_pred, zero_division=0)
        print(f"{name} test accuracy for {label}: {acc:.3f}")
        print(f"{name} test recall for {label}: {rec:.3f}")
        print(f"{name} test precision for {label}: {prec:.3f}")

### 1.3.7 - Mapper ROC curve function definition

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt
import math

def show_roc_mapper(mapper_dict, name):
    prob_dict = {}

    n_plots = len(mapper_dict)
    n_cols = 3
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows)
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, (label, predictor) in zip(axes, mapper_dict.items()):
        y_true = y_test_dict[label]
        y_prob = predictor.predict_proba(X_test_dict[label])[:, 1]
        prob_dict[label] = y_prob

        if len(np.unique(y_true)) < 2:
            print(f"Skipping ROC for {name} / {label}: fewer than 2 unique labels in test set")
            ax.text(0.5, 0.5, "Single class in test set\n(ROC undefined)", ha="center", va="center")
            ax.set_title(f"{label} (skipped)")
            ax.grid(True)
            continue

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = roc_auc_score(y_true, y_prob)

        RocCurveDisplay(
            fpr=fpr,
            tpr=tpr,
            roc_auc=roc_auc,
            estimator_name=f"{name} {label}"
        ).plot(ax=ax)

        ax.set_title(f"{label}")
        ax.grid(True)

    # Hide any unused subplots
    for ax in axes[n_plots:]:
        ax.set_visible(False)

    fig.suptitle(f"ROC Curves ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOTS_DIR / f"roc_{name}_mapper.png", dpi=300, bbox_inches="tight")

    plt.show()

    return prob_dict

### 1.3.8 - Mapper Precision-Recall curve function definition

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    PrecisionRecallDisplay,
)
import matplotlib.pyplot as plt
import math

def show_prc_mapper(mapper_dict, name):
    prob_dict = {}

    n_plots = len(mapper_dict)
    n_cols = 3
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows)
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, (label, predictor) in zip(axes, mapper_dict.items()):
        y_true = y_test_dict[label]
        y_prob = predictor.predict_proba(X_test_dict[label])[:, 1]
        prob_dict[label] = y_prob

        if len(np.unique(y_true)) < 2:
            print(f"Skipping PRC for {name} / {label}: fewer than 2 unique labels in test set")
            ax.text(0.5, 0.5, "Single class in test set\n(PRC undefined)", ha="center", va="center")
            ax.set_title(f"{label} (skipped)")
            ax.grid(True)
            continue

        pre, rec, _ = precision_recall_curve(y_true, y_prob)
        avg_pre = average_precision_score(y_true, y_prob)

        PrecisionRecallDisplay(
            precision=pre,
            recall=rec,
            average_precision=avg_pre,
            estimator_name=f"{name} {label}"
        ).plot(ax=ax)

        ax.set_title(f"{label}")
        ax.grid(True)

    for ax in axes[n_plots:]:
        ax.set_visible(False)

    fig.suptitle(f"Precision-Recall Curves ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOTS_DIR / f"prc_{name}_mapper.png", dpi=300, bbox_inches="tight")

    plt.show()

    return prob_dict

### 1.3.9 - Test attack type mappers

In [ ]:
evaluate_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
show_roc_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
show_prc_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
evaluate_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
show_roc_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
show_prc_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
evaluate_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
show_roc_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
show_prc_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
evaluate_mapper(logistic_regression_dict,"LogisticRegression")

In [ ]:
show_roc_mapper(logistic_regression_dict,"LogisticRegression")

In [ ]:
show_prc_mapper(logistic_regression_dict,"LogisticRegression")

### 1.3.10 - Computing attack type score

In [ ]:
xgb_attack_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, xgboost in xgb_dict.items():
    y_prob = xgboost.predict_proba(X_test_dict[label])[:, 1]
    xgb_attack_type_prob[label]=y_prob

In [ ]:
for key, prob_vector in xgb_attack_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

In [ ]:
gnb_attack_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, gaussiannb in naive_bayes_dict.items():
    y_prob = gaussiannb.predict_proba(X_test_dict[label])[:, 1]
    gnb_attack_type_prob[label]=y_prob

In [ ]:
for key, prob_vector in gnb_attack_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

In [ ]:
lr_attack_type_prob = {}

In [ ]:
for label, lr_model in logistic_regression_dict.items():
    y_prob = lr_model.predict_proba(X_test_dict[label])[:, 1]
    lr_attack_type_prob[label] = y_prob

In [ ]:
for key, prob_vector in lr_attack_type_prob.items():
    print(key + f":{prob_vector.shape[0]}")

In [ ]:
perfect_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, score in y_test_dict.items():
    perfect_type_prob[label] = np.asarray(score)

In [ ]:
for key, prob_vector in perfect_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

## 1.4 - Building the scenario reconstruction dataset

### 1.4.1 - Loading the starting dataset 

In [ ]:
import numpy as np
import pandas as pd

df_rec = pd.read_csv(MERGED_DATA_ROOT / "test.csv", low_memory=False)
df_rec["index"] = np.arange(len(df_rec))
df_rec.info()

In [ ]:
df_rec = df_rec.dropna(thresh=2, axis=0)

In [ ]:
df_rec = df_rec[df_rec["label"].isin(df_train["label"].unique())]
df_rec = df_rec[df_rec["label"].isin(df_test["label"].unique())]

In [ ]:
attack_steps = (
    df_rec
    .groupby(["campaign_id", "label"], as_index=False)
    .agg(total_steps=("step_number", "nunique"))
)

print(attack_steps)

In [ ]:
print(df_rec.shape[0])

### 1.4.2 - Further step subdivision

In [ ]:
attack = df_rec["step_number"] != -1
groups = (
    df_rec.loc[attack]
      .groupby(["campaign_id", "step_number", "src_ip", "dst_ip"], sort=False)
      .ngroup()
)
df_rec.loc[attack, "step_number"] = groups

In [ ]:
attack_steps = (
    df_rec[df_rec["label"] != "normal"]
    .groupby(["campaign_id", "label"], as_index=False)
    .agg(total_steps=("step_number", "nunique"))
)

In [ ]:
attack_steps

In [ ]:
df = df_rec.sort_values("timestamp").reset_index(drop=True)
df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)
df = df[df["label"] != "normal"]

step_summary = (
    df.groupby("step_number", as_index=False)
      .agg(
          campaign_id  = ("campaign_id", "first"),
          label        = ("label", "first"),
          num_flws     = ("timestamp", "size"),
          start_time   = ("timestamp", "min"),
          end_time     = ("timestamp", "max")
      )
      .assign(duration=lambda g: g["end_time"] - g["start_time"])
)

step_summary["fps"] = (
    step_summary["num_flws"] /
    step_summary["duration"].dt.total_seconds().replace(0, np.nan)
)

label_summary = (
    step_summary.groupby(["campaign_id", "label"], as_index=False)
               .agg(
                   avg_num_flws    = ("num_flws", "mean"),
                   median_num_flws = ("num_flws", "median"),
                   avg_duration    = ("duration", "mean"),
                   median_duration = ("duration", "median"),
                   total_steps     = ("step_number", "nunique"),
                   total_flws      = ("num_flws", "sum"),
                   avg_fps         = ("fps", "mean")
               )
)

In [ ]:
step_summary

In [ ]:
label_summary

### 1.4.3 - Extracting reconstruction features

In [ ]:
rec_cols = [
    "src_ip", "src_port", "dst_ip", "dst_port", "protocol",
    "timestamp", "bidirectional_duration_ms", "campaign_id",
    "label", "step_number", "index",
]
df_rec = df_rec[rec_cols].rename(columns={"bidirectional_duration_ms": "duration"}).copy()

In [ ]:
print(df_rec.shape[0])

### 1.4.4 - Adding new features

In [ ]:
mapper_probs = {
    "xgb": xgb_attack_type_prob,
    "gnb": gnb_attack_type_prob,
    "lr": lr_attack_type_prob,
    "perfect": perfect_type_prob,
}

ind_test_arr = np.asarray(ind_test)
df_rec_dict = {}
for anomaly_name, anomaly_scores in anomaly_score_dict.items():
    anomaly_df = pd.DataFrame({
        "index": ind_test_arr,
        "anomaly_score": np.asarray(anomaly_scores),
    })
    base_df = df_rec.merge(anomaly_df, on="index", how="inner", validate="one_to_one")
    assert len(base_df) == len(df_rec) == len(ind_test_arr), (
        f"Index alignment mismatch for {anomaly_name}: "
        f"merged={len(base_df)}, df_rec={len(df_rec)}, ind_test={len(ind_test_arr)}"
    )
    for mapper_name, type_probs in mapper_probs.items():
        type_df = pd.DataFrame({
            "index": ind_test_arr,
            **{attack_type: np.asarray(probs) for attack_type, probs in type_probs.items()},
        })
        merged_df = base_df.merge(type_df, on="index", how="inner", validate="one_to_one")
        key = f"{anomaly_name}_{mapper_name}"
        df_rec_dict[key] = merged_df
        print(f"{key}: shape={merged_df.shape}")

### 1.4.5 - Saving the dataset 

In [ ]:
EXPORT_POOLED_COPY = True

for campaign_name, cid in campaign_ids.items():
    campaign_dir = FINAL_DATA_ROOT / campaign_name
    campaign_dir.mkdir(parents=True, exist_ok=True)
    for name, df in df_rec_dict.items():
        campaign_df = (
            df[(df["campaign_id"] == cid) | (df["label"] == "normal")]
            .sort_values(by="timestamp")
        )
        campaign_df.to_csv(campaign_dir / f"{name}_rec_dataset.csv", index=False)
    print(f"Exported {len(df_rec_dict)} datasets for '{campaign_name}' (campaign_id={cid}) to {campaign_dir}")

if EXPORT_POOLED_COPY:
    pooled_dir = FINAL_DATA_ROOT / "all_campaigns"
    pooled_dir.mkdir(parents=True, exist_ok=True)
    for name, df in df_rec_dict.items():
        df.sort_values(by="timestamp").to_csv(pooled_dir / f"{name}_rec_dataset.csv", index=False)
    print(f"Exported {len(df_rec_dict)} pooled datasets to {pooled_dir}")